# 00 — Smoke test de Free Edition
**Spec:** docs/specs/00_entorno.md · **Requisitos:** R-37, R-17, R-28 · **Escribe:** `iol_ops.smoke_test_results`

Verifica lo que puede cambiar el plan: orquestación con `dbutils.notebook.run`, salida a internet hacia los
proveedores de cotizaciones (ADR-008), `ai_query` (ADR-011), liquid clustering en `CREATE` y `ALTER` (ADR-012), región y el MERGE Delta de `src/`.
Ningún chequeo corta el notebook: cada uno registra OK, INCONCLUSO o FAIL con el detalle. Pasar la tabla final a ADR-001/008/011.

In [0]:
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("ai_model", "")

import json
import os
import sys
import uuid
from datetime import datetime, timezone

sys.path.insert(0, os.path.abspath("../src"))
from iol_pipeline import config, io, smoke

CATALOG = dbutils.widgets.get("catalog")
SMOKE_RUN_ID = str(uuid.uuid4())
results = []


def record(check: str, status: str, detail: str) -> None:
    results.append({"check": check, "status": status, "detail": detail[:config.SMOKE_DETAIL_MAX_CHARS],
                    "checked_at": datetime.now(timezone.utc).isoformat(timespec="seconds")})
    print(f"[{status}] {check}: {detail[:config.SMOKE_PRINT_MAX_CHARS]}")


io.ensure_schemas(spark, catalog=CATALOG)
print("smoke_run_id:", SMOKE_RUN_ID, "· DBR:", os.environ.get("DATABRICKS_RUNTIME_VERSION"), "· Spark:", spark.version)

smoke_run_id: d96b79bc-649b-483d-b9ca-aa87e463b212 · DBR: client.6.5 · Spark: 4.2.0


## 1. Orquestación: `dbutils.notebook.run`

In [0]:
try:
    raw = dbutils.notebook.run("./00_smoke_test_child", config.SMOKE_CHILD_TIMEOUT_S, {"ping": SMOKE_RUN_ID})
    echo = json.loads(raw).get("echo")
    record("orchestration:notebook_run", "OK" if echo == SMOKE_RUN_ID else "FAIL", f"child devolvió {raw}")
except Exception as exc:
    record("orchestration:notebook_run", "FAIL", f"{type(exc).__name__}: {exc}")

[OK] orchestration:notebook_run: child devolvió {"status": "SUCCESS", "echo": "d96b79bc-649b-483d-b9ca-aa87e463b212"}


## 2. Egress a proveedores de cotizaciones (ADR-008)
HTTP 403, 407 o 429 = INCONCLUSO (pudo responder un intermediario). Otra respuesta HTTP = host alcanzable. Error de conexión o timeout = bloqueado.

In [0]:
import requests

for r in smoke.check_egress(config.SMOKE_EGRESS_URLS, requests.get, config.HTTP_TIMEOUT_S,
                            config.SMOKE_EGRESS_INCONCLUSIVE_HTTP):
    record(r["check"], r["status"], r["detail"])

[OK] egress:iol_api: https://api.invertironline.com -> HTTP 200, Server=cloudflare
[OK] egress:byma_open_data: https://open.bymadata.com.ar -> HTTP 200, Server=nginx
[OK] egress:data912: https://data912.com -> HTTP 200, Server=cloudflare
[INCONCLUSO] egress:yahoo_finance: https://query1.finance.yahoo.com -> HTTP 429, Server=ATS


## 3. IA: endpoints disponibles y `ai_query` (ADR-011)

In [0]:
endpoints = []
try:
    from databricks.sdk import WorkspaceClient

    endpoints = sorted(e.name for e in WorkspaceClient().serving_endpoints.list())
    record("ai:serving_endpoints", "OK" if endpoints else "FAIL", ", ".join(endpoints) or "ninguno visible")
except Exception as exc:
    record("ai:serving_endpoints", "FAIL", f"{type(exc).__name__}: {exc}")

model = dbutils.widgets.get("ai_model") or config.SMOKE_AI_MODEL
try:
    answer = spark.sql(f"SELECT ai_query('{model}', 'Respond with the single word: pong') AS r").first()["r"]
    record(f"ai:ai_query:{model}", "OK", str(answer))
except Exception as exc:
    record(f"ai:ai_query:{model}", "FAIL", f"{type(exc).__name__}: {exc}")

[OK] ai:serving_endpoints: databricks-bge-large-en, databricks-deepseek-v4-flash-0731, databricks-gemma-3-12b, databricks-gpt-oss-120b, databricks-gpt-oss-20b, databricks-gte-large-en, databricks-llama-4-maverick, databricks-meta-llama-3-1-8b-instruct, databricks-meta-llama-3-3-70b-instruct, databricks-qwen3-embedding-0-6b, d
[OK] ai:ai_query:databricks-meta-llama-3-3-70b-instruct: pong


## 4. Liquid clustering (ADR-012)

In [0]:
probe = io.table_name("ops", "smoke_cluster_probe", catalog=CATALOG)
try:
    spark.sql(f"CREATE OR REPLACE TABLE {probe} (id INT, d DATE) CLUSTER BY (d)")
    clustering = spark.sql(f"DESCRIBE DETAIL {probe}").first()["clusteringColumns"]
    record("delta:liquid_clustering", "OK" if list(clustering) == ["d"] else "FAIL", f"clusteringColumns={clustering}")
except Exception as exc:
    record("delta:liquid_clustering", "FAIL", f"{type(exc).__name__}: {exc}")
finally:
    spark.sql(f"DROP TABLE IF EXISTS {probe}")

# El antes y después del spec 07 agrega clustering a una tabla que ya existe.
probe_alter = io.table_name("ops", "smoke_alter_cluster_probe", catalog=CATALOG)
try:
    spark.sql(f"CREATE OR REPLACE TABLE {probe_alter} (id INT, d DATE)")
    spark.sql(f"ALTER TABLE {probe_alter} CLUSTER BY (d)")
    clustering = spark.sql(f"DESCRIBE DETAIL {probe_alter}").first()["clusteringColumns"]
    record("delta:alter_cluster_by", "OK" if list(clustering) == ["d"] else "FAIL", f"clusteringColumns={clustering}")
except Exception as exc:
    record("delta:alter_cluster_by", "FAIL", f"{type(exc).__name__}: {exc}")
finally:
    spark.sql(f"DROP TABLE IF EXISTS {probe_alter}")

[OK] delta:liquid_clustering: clusteringColumns=['d']
[OK] delta:alter_cluster_by: clusteringColumns=['d']


## 5. Región y catálogo por defecto

In [0]:
try:
    # current_metastore() = <cloud>:<region>:<uuid>; el uuid no se guarda.
    row = spark.sql("SELECT current_metastore() AS m, current_catalog() AS c").first()
    record("env:region_catalog", "OK", f"{':'.join(row['m'].split(':')[:2])} · catálogo por defecto: {row['c']}")
except Exception as exc:
    record("env:region_catalog", "FAIL", f"{type(exc).__name__}: {exc}")

[OK] env:region_catalog: aws:us-east-2 · catálogo por defecto: workspace


## 6. MERGE Delta de `src/iol_pipeline/io.py` en serverless + resultados persistidos

In [0]:
from pyspark.sql import functions as F

target = io.table_name("ops", "smoke_test_results", catalog=CATALOG)
keys = ["smoke_run_id", "check"]


def to_df(rows):
    return (spark.createDataFrame(rows, "check string, status string, detail string, checked_at string")
            .select(F.lit(SMOKE_RUN_ID).alias("smoke_run_id"), "check", "status", "detail",
                    F.to_timestamp("checked_at").alias("checked_at")))


try:
    first = io.write_delta_merge(spark, to_df(results), target, keys)
    rerun = io.write_delta_merge(spark, to_df(results), target, keys)
    idempotent = first["rows_inserted"] == len(results) and rerun["rows_inserted"] == 0
    record("delta:write_delta_merge_idempotent", "OK" if idempotent else "FAIL", f"1ra corrida={first} · 2da corrida={rerun}")
except Exception as exc:
    record("delta:write_delta_merge_idempotent", "FAIL", f"{type(exc).__name__}: {exc}")

try:
    io.write_delta_merge(spark, to_df(results[-1:]), target, keys)
except Exception as exc:
    record("delta:persist_results", "FAIL", f"{type(exc).__name__}: {exc}")

[OK] delta:write_delta_merge_idempotent: 1ra corrida={'rows_source': 10, 'rows_inserted': 10, 'rows_already_present': 0} · 2da corrida={'rows_source': 10, 'rows_inserted': 0, 'rows_already_present': 10}


## Resultado (copiar a ADR-001 / ADR-008 / ADR-011)

In [0]:
display(to_df(results).orderBy("check"))  # desde memoria: se ve aunque el MERGE haya fallado
failed = [r["check"] for r in results if r["status"] == "FAIL"]
inconclusive = [r["check"] for r in results if r["status"] == "INCONCLUSO"]
print(f"{len(results) - len(failed) - len(inconclusive)}/{len(results)} OK · "
      f"INCONCLUSO: {inconclusive or 'ninguno'} · FAIL: {failed or 'ninguno'}")
status = smoke.run_status(results)
print("status:", status)

smoke_run_id,check,status,detail,checked_at
d96b79bc-649b-483d-b9ca-aa87e463b212,ai:ai_query:databricks-meta-llama-3-3-70b-instruct,OK,pong,2026-11-01T14:21:29.000Z
d96b79bc-649b-483d-b9ca-aa87e463b212,ai:serving_endpoints,OK,"databricks-bge-large-en, databricks-deepseek-v4-flash-0731, databricks-gemma-3-12b, databricks-gpt-oss-120b, databricks-gpt-oss-20b, databricks-gte-large-en, databricks-llama-4-maverick, databricks-meta-llama-3-1-8b-instruct, databricks-meta-llama-3-3-70b-instruct, databricks-qwen3-embedding-0-6b, databricks-qwen3-next-80b-a3b-instruct, databricks-qwen35-122b-a10b",2026-11-01T14:21:25.000Z
d96b79bc-649b-483d-b9ca-aa87e463b212,delta:alter_cluster_by,OK,clusteringColumns=['d'],2026-11-01T14:21:39.000Z
d96b79bc-649b-483d-b9ca-aa87e463b212,delta:liquid_clustering,OK,clusteringColumns=['d'],2026-11-01T14:21:35.000Z
d96b79bc-649b-483d-b9ca-aa87e463b212,delta:write_delta_merge_idempotent,OK,"1ra corrida={'rows_source': 10, 'rows_inserted': 10, 'rows_already_present': 0} · 2da corrida={'rows_source': 10, 'rows_inserted': 0, 'rows_already_present': 10}",2026-11-01T14:21:51.000Z
d96b79bc-649b-483d-b9ca-aa87e463b212,egress:byma_open_data,OK,"https://open.bymadata.com.ar -> HTTP 200, Server=nginx",2026-11-01T14:21:24.000Z
d96b79bc-649b-483d-b9ca-aa87e463b212,egress:data912,OK,"https://data912.com -> HTTP 200, Server=cloudflare",2026-11-01T14:21:24.000Z
d96b79bc-649b-483d-b9ca-aa87e463b212,egress:iol_api,OK,"https://api.invertironline.com -> HTTP 200, Server=cloudflare",2026-11-01T14:21:24.000Z
d96b79bc-649b-483d-b9ca-aa87e463b212,egress:yahoo_finance,INCONCLUSO,"https://query1.finance.yahoo.com -> HTTP 429, Server=ATS",2026-11-01T14:21:24.000Z
d96b79bc-649b-483d-b9ca-aa87e463b212,env:region_catalog,OK,aws:us-east-2 · catálogo por defecto: workspace,2026-11-01T14:21:40.000Z


10/11 OK · INCONCLUSO: ['egress:yahoo_finance'] · FAIL: ninguno
status: SUCCESS


In [0]:
dbutils.notebook.exit(json.dumps({"status": status, "smoke_run_id": SMOKE_RUN_ID, "failed_checks": failed,
                                 "inconclusive_checks": inconclusive}))